# Section 09 — Intrinsic Hierarchy Evaluation

This notebook evaluates the quality of the generated multiresolution temporal
hypergraph hierarchy without using external benchmark questions.

The evaluation focuses on intrinsic properties:

1. semantic coherence of supernodes;
2. hypergraph-native structural quality;
3. hierarchy consistency;
4. temporal identity preservation.

The goal is to determine whether the hierarchy is internally meaningful before
performing downstream QA evaluation.

In [1]:
from pathlib import Path

REPO_DIR = Path(
    "/content/tkh-hierarchy-project"
)

if not REPO_DIR.exists():
    !git clone https://github.com/mohamadghoroobi/tkh-hierarchy-project.git

%cd /content/tkh-hierarchy-project

Cloning into 'tkh-hierarchy-project'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 127 (delta 70), reused 85 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 17.08 MiB | 24.49 MiB/s, done.
Resolving deltas: 100% (70/70), done.
/content/tkh-hierarchy-project


## Imports

In [ ]:
from pathlib import Path
import json
import math

from collections import defaultdict, Counter

import numpy as np
import pandas as pd

## Paths

In [ ]:
PROJECT_DIR = Path(
    "/content/tkh-hierarchy-project"
)


LABEL_DIR = (
    PROJECT_DIR
    /
    "artifacts"
    /
    "hierarchy"
    /
    "labelled"
)


TEMPORAL_DIR = (
    PROJECT_DIR
    /
    "artifacts"
    /
    "hierarchy"
    /
    "temporal"
)


EVAL_DIR = (
    PROJECT_DIR
    /
    "artifacts"
    /
    "evaluation"
)


EVAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


SNAPSHOT_YEARS = [
    2020,
    2022,
    2024,
    2026
]


LEVELS = [
    0,
    1,
    2
]

## Load labelled hierarchies

In [ ]:
def load_labelled_hierarchy(year):

    path = (
        LABEL_DIR
        /
        f"hierarchy_{year}_labelled.json"
    )


    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)



labelled_hierarchies = {

    year:
        load_labelled_hierarchy(year)

    for year
    in SNAPSHOT_YEARS
}

## Load temporal events

In [ ]:
with open(
    TEMPORAL_DIR /
    "temporal_events.json",
    "r",
    encoding="utf-8"
) as f:

    temporal_events = json.load(f)


print(
    "Events:",
    len(temporal_events)
)

Events: 2081


## Evaluation dimensions

For a hierarchy level $P_k$, each supernode $C$ is evaluated using:

### Semantic coherence

Whether members inside a supernode share similar semantic concepts.

### Structural quality

Whether native hyperedges remain internally meaningful and avoid excessive
fragmentation.

### Hierarchy consistency

Whether parent-child relationships preserve the expected multiresolution
organization.

### Temporal stability

Whether persistent identities remain coherent across snapshots.

No gold labels or benchmark answers are used in this section.

## Semantic coherence evaluation

We use the labels/evidence generated in Section 08.

A simple intrinsic measure:

For each cluster:

$$ SC(C)= \frac{ |\text{unique evidence terms}| }{ |\text{members}| } $$

Lower diversity of concepts indicates stronger coherence.

## Semantic diversity

In [ ]:
def semantic_coherence_score(
    record
):

    evidence = (
        record.get(
            "label_evidence",
            []
        )
    )


    size = len(
        record[
            "member_ids"
        ]
    )


    if size == 0:

        return 0


    return (
        1
        -
        min(
            len(set(evidence))
            /
            size,
            1
        )
    )

## Calculate semantic scores

In [ ]:
semantic_rows = []


for year, hierarchy in labelled_hierarchies.items():

    for level in LEVELS:


        for record in (
            hierarchy["levels"][str(level)]
        ):

            semantic_rows.append({

                "year":
                    year,

                "level":
                    level,

                "persistent_id":
                    record["persistent_id"],

                "label":
                    record.get(
                        "label"
                    ),

                "semantic_coherence":
                    semantic_coherence_score(
                        record
                    )
            })



semantic_df = pd.DataFrame(
    semantic_rows
)


semantic_df.head()

,year,level,persistent_id,label,semantic_coherence
0,2020,0,L0_C00000,Prediction,0.982111
1,2020,0,L0_C00001,Materials,0.952381
2,2020,0,L0_C00002,Carbon,0.938272
3,2020,0,L0_C00003,Wang,0.888889
4,2020,0,L0_C00004,Magpie,0.900000


## Hypergraph structural evaluation

We measure hyperedge fragmentation.

For a hyperedge:

$$ p_{e,c} = \frac{|e\cap C_c|} {|e|} $$

Entropy:

$$ H(e) = - \sum_c p_{e,c}\log(p_{e,c}) $$

Normalized:

$$ H_n(e) = \frac{H(e)} {\log(K)} $$

Lower is better.

## Load original TKH

In [ ]:
with open(
    PROJECT_DIR /
    "data" /
    "tkh_collection10.json",
    "r",
    encoding="utf-8"
) as f:

    tkh = json.load(f)


nodes = tkh["nodes"]

hyperedges = tkh["hyperedges"]

## Build node assignments

In [ ]:
def node_assignment(
    hierarchy,
    level
):

    result = {}


    for cluster in (
        hierarchy["levels"][str(level)]
    ):

        for node in cluster["member_ids"]:

            result[node] = (
                cluster["persistent_id"]
            )


    return result

## Hyperedge entropy

In [ ]:
def hyperedge_entropy(
    edge_nodes,
    assignment
):

    counts = Counter()


    for node in edge_nodes:

        if node in assignment:

            counts[
                assignment[node]
            ] += 1


    if len(counts) <= 1:

        return 0.0


    total = sum(
        counts.values()
    )


    entropy = 0


    for count in counts.values():

        p = count / total

        entropy -= (
            p *
            math.log(p)
        )


    return entropy / math.log(
        len(counts)
    )

## Compute structural fragmentation

In [ ]:
structural_rows = []


for year, hierarchy in labelled_hierarchies.items():

    for level in LEVELS:


        assignment = node_assignment(
            hierarchy,
            level
        )


        scores = []


        for edge in hyperedges:

            scores.append(
                hyperedge_entropy(
                    edge["members"],
                    assignment
                )
            )


        structural_rows.append({

            "year":
                year,

            "level":
                level,

            "mean_hyperedge_entropy":
                np.mean(scores),

            "median_hyperedge_entropy":
                np.median(scores)
        })



structural_df = pd.DataFrame(
    structural_rows
)


structural_df

,year,level,mean_hyperedge_entropy,median_hyperedge_entropy
0,2020,0,0.055958,0.0
1,2020,1,0.068896,0.0
2,2020,2,0.084603,0.0
3,2022,0,0.056167,0.0
4,2022,1,0.083540,0.0
5,2022,2,0.093567,0.0
6,2024,0,0.040072,0.0
7,2024,1,0.068357,0.0
8,2024,2,0.094760,0.0
9,2026,0,0.041700,0.0


## Hierarchy consistency

We verify:

every child has a valid parent;
every node belongs exactly once;
parent-child counts are consistent.

In [ ]:
hierarchy_rows = []


for year, hierarchy in labelled_hierarchies.items():

    for level in [1,2]:


        parents = {

            r["persistent_id"]

            for r
            in hierarchy["levels"][str(level-1)]
        }


        valid = True


        for child in (
            hierarchy["levels"][str(level)]
        ):

            if (
                child[
                    "parent_persistent_id"
                ]
                not in parents
            ):

                valid = False



        hierarchy_rows.append({

            "year":
                year,

            "level":
                level,

            "parent_links_valid":
                valid

        })



hierarchy_df = pd.DataFrame(
    hierarchy_rows
)


hierarchy_df

,year,level,parent_links_valid
0,2020,1,True
1,2020,2,True
2,2022,1,True
3,2022,2,True
4,2024,1,True
5,2024,2,True
6,2026,1,True
7,2026,2,True


## Temporal diagnostics

In [ ]:
event_summary = (
    Counter(
        e["event_type"]
        for e
        in temporal_events
    )
)


event_summary

Counter({'birth': 716,
         'death': 344,
         'split': 277,
         'merge': 310,
         'growth': 434})

## Combine evaluation report

In [ ]:
evaluation_report = {

    "semantic":

        semantic_df.to_dict(
            orient="records"
        ),


    "structural":

        structural_df.to_dict(
            orient="records"
        ),


    "hierarchy":

        hierarchy_df.to_dict(
            orient="records"
        ),


    "temporal_events":

        dict(
            event_summary
        )
}

## Save report

In [ ]:
OUTPUT = (
    EVAL_DIR
    /
    "intrinsic_evaluation_report.json"
)


with open(
    OUTPUT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        evaluation_report,
        f,
        indent=2
    )


print(
    "Saved:",
    OUTPUT
)

Saved: /content/tkh-hierarchy-project/artifacts/evaluation/intrinsic_evaluation_report.json


## Final summary

In [ ]:
print(
    "===== SECTION 09 COMPLETE ====="
)

print(
    "Semantic records:",
    len(semantic_df)
)

print(
    "Structural evaluations:",
    len(structural_df)
)

print(
    "Hierarchy checks:",
    len(hierarchy_df)
)

print(
    "Temporal events:",
    len(temporal_events)
)

===== SECTION 09 COMPLETE =====
Semantic records: 1488
Structural evaluations: 12
Hierarchy checks: 8
Temporal events: 2081


# Section 8 has established


*   The multi-resolution hierarchy created in previous stages has been enriched with semantic labels and descriptions.
*   Abstract clusters are no longer represented only by structural identifiers; they are associated with human-interpretable concepts.
*   Labels were generated using information from the entities contained within each hierarchical group.
*   The labelling process preserves the connection between:
    *   original TKH entities;
    *   abstract clusters;
    *   temporal hierarchy identities.

*   Labels provide an interpretable representation of higher-level knowledge structures while maintaining traceability to the underlying hypergraph.


---

## Contribution to the project

Before labelling:
```text
Multi-resolution hierarchy

L0_C00003
L1_C00015
L2_C00120
```

After labelling:

## Git Push

In [ ]:
from pathlib import Path

CLEAN = Path("/content/drive/MyDrive/Apply/Germany/ConstructorLabs/09_intrinsic_evaluation.ipynb")
REPO_FILE = Path(
    "/content/tkh-hierarchy-project/"
    "notebooks/09_intrinsic_evaluation.ipynb"
)

print("Clean file exists:", CLEAN.exists())
print("Repo file exists :", REPO_FILE.exists())

Clean file exists: True
Repo file exists : False


In [ ]:
import shutil

shutil.copy2(CLEAN, REPO_FILE)

print("Clean notebook copied into repository.")

Clean notebook copied into repository.


In [ ]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	artifacts/evaluation/
	notebooks/09_intrinsic_evaluation.ipynb

nothing added to commit but untracked files present (use "git add" to track)


In [ ]:
!git add -A

In [ ]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   artifacts/evaluation/intrinsic_evaluation_report.json
	new file:   notebooks/09_intrinsic_evaluation.ipynb



In [ ]:
!git config --global user.name "mohamadghoroobi"
!git config --global user.email "m.ghoroobi@gmail.com"

In [ ]:
commit_message = """ feat(evaluation): add intrinsic hierarchy quality evaluation" \

Implement intrinsic evaluation metrics for the temporally coupled multiresolution hypergraph hierarchy.

- load labelled temporal hierarchies from Section 08

- validate hierarchy inputs before evaluation

- evaluate semantic coherence of generated supernode labels

- measure hypergraph structural quality using native hyperedge entropy

- compute hyperedge fragmentation without pairwise graph projection

- preserve original TKH hypergraph membership structure during evaluation

- validate parent-child hierarchy consistency across levels

- analyze temporal evolution events from persistent supernode identities

- generate semantic, structural, hierarchical, and temporal evaluation reports

- export intrinsic evaluation results for downstream analysis

- document evaluation methodology and limitations
"""

with open("/tmp/commit_message.txt", "w", encoding="utf-8") as f:
    f.write(commit_message)

In [ ]:
!git commit -F /tmp/commit_message.txt

[main c80cbeb]  feat(evaluation): add intrinsic hierarchy quality evaluation" Implement intrinsic evaluation metrics for the temporally coupled multiresolution hypergraph hierarchy.
 2 files changed, 10544 insertions(+)
 create mode 100644 artifacts/evaluation/intrinsic_evaluation_report.json
 create mode 100644 notebooks/09_intrinsic_evaluation.ipynb


In [ ]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

assert token, "GITHUB_TOKEN not found"
print("Token loaded successfully")

Token loaded successfully


In [ ]:
import os
import subprocess
from pathlib import Path

username = "mohamadghoroobi"

env = os.environ.copy()

env["GITHUB_USER"] = "mohamadghoroobi"
env["GITHUB_TOKEN"] = token
env["GIT_TERMINAL_PROMPT"] = "0"


askpass = Path("/tmp/git_askpass.sh")

askpass.write_text(
"""#!/bin/sh
case "$1" in
  *Username*) echo "$GITHUB_USER" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
"""
)

askpass.chmod(0o700)

env["GIT_ASKPASS"] = str(askpass)

print("Git authentication prepared")

Git authentication prepared


In [ ]:
subprocess.run(
    ["git", "push", "origin", "main"],
    env=env,
    check=True
)

print("Push completed successfully")

Push completed successfully


In [ ]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
